# DORA Metrics for Lead Time Only

## Env

**Output:**
- Working Kernel
- Loaded python libraries

In [1]:
print("Kernel is working.")

Kernel is working.


In [2]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Data Ingestion

**Output:** pandas dataframe *df_lead*.

In [3]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/change_lead_time.json'

# Try reading with explicit encoding and error handling
try:
    df_lead = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"]).sort_values(by=["commit_time"])
    print("\nDataframe info:")
    print(df_lead.info())
    print("\nFirst 5 rows:")
    print(df_lead.head())
except Exception as e:
    print(f"Error: {str(e)}")


Dataframe info:
<class 'pandas.core.frame.DataFrame'>
Index: 44 entries, 28 to 5
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   change_id        44 non-null     object             
 1   application_id   44 non-null     object             
 2   commit_id        44 non-null     object             
 3   commit_time      44 non-null     datetime64[ns, UTC]
 4   deploy_id        44 non-null     object             
 5   deploy_time      44 non-null     datetime64[ns, UTC]
 6   lead_time_hours  44 non-null     int64              
dtypes: datetime64[ns, UTC](2), int64(1), object(4)
memory usage: 2.8+ KB
None

First 5 rows:
       change_id application_id commit_id               commit_time  \
28  chg_a5f8cfaf         app003    bdc28e 2025-07-01 08:09:00+00:00   
35  chg_dd68cbb5         app003    4bb702 2025-07-02 00:37:00+00:00   
8   chg_8f6ee15b         app001    ca3e35 2025-07-07 13:3

## Data Manipulation

**Output:** Updates to pandas data frame *df_lead* for use by figures.
- New aggregated month datetime column.
- New Moving Average columns for plotting (SMA, EMA, CMA)

In [4]:
# Add new row and quater columns
df_lead["commit_month"] = df_lead["commit_time"].dt.month
df_lead["deploy_month"] = df_lead["deploy_time"].dt.month

df_lead.sort_values(by=["commit_time"], inplace=True)

df_lead.head(100)

,change_id,application_id,commit_id,commit_time,deploy_id,deploy_time,lead_time_hours,commit_month,deploy_month
28,chg_a5f8cfaf,app003,bdc28e,2025-07-01 08:09:00+00:00,df_c5a7c423,2025-07-02 16:09:00+00:00,32,7,7
35,chg_dd68cbb5,app003,4bb702,2025-07-02 00:37:00+00:00,df_15eb7c8c,2025-07-02 10:37:00+00:00,10,7,7
8,chg_8f6ee15b,app001,ca3e35,2025-07-07 13:35:00+00:00,df_295ed3f1,2025-07-07 16:35:00+00:00,3,7,7
11,chg_90d9652e,app001,cd7d15,2025-07-10 17:58:00+00:00,df_83590aea,2025-07-13 00:58:00+00:00,55,7,7
40,chg_22754b79,app003,00540b,2025-07-13 02:40:00+00:00,df_d6b6cdf5,2025-07-16 01:40:00+00:00,71,7,7
12,chg_cfd655f7,app001,e65c95,2025-07-13 05:43:00+00:00,df_f8b93c0a,2025-07-15 07:43:00+00:00,50,7,7
16,chg_1918cc5e,app002,2e6951,2025-07-18 22:43:00+00:00,df_a29aae73,2025-07-19 08:43:00+00:00,10,7,7
14,chg_db7dbdff,app002,3d91ec,2025-07-20 13:15:00+00:00,df_8966166b,2025-07-22 04:15:00+00:00,39,7,7
2,chg_5e11ee73,app001,966c88,2025-07-25 05:08:00+00:00,df_45056a55,2025-07-27 12:08:00+00:00,55,7,7
10,chg_2c869d04,app001,4c513e,2025-07-27 19:14:00+00:00,df_325b9203,2025-07-30 06:14:00+00:00,59,7,7


In [19]:
# Add moving average column
# https://www.geeksforgeeks.org/pandas/how-to-calculate-moving-average-in-a-pandas-dataframe/

# Simple Moving Average
df_lead["SMA"] = df_lead["lead_time_hours"].rolling(window=30).mean()

# Exponential Moving Average
df_lead["EMA"] = df_lead["lead_time_hours"].ewm(span=30, adjust=True).mean()

# Cumulative Moving Average
df_lead["CMA"] = df_lead["lead_time_hours"].expanding().mean()

df_lead.sort_values(by=["commit_time"], inplace=True)

df_lead.head(100)

,change_id,application_id,commit_id,commit_time,deploy_id,deploy_time,lead_time_hours,commit_month,deploy_month,SMA,EMA,CMA
28,chg_a5f8cfaf,app003,bdc28e,2025-07-01 08:09:00+00:00,df_c5a7c423,2025-07-02 16:09:00+00:00,32,7,7,NaN,32.000000,32.000000
35,chg_dd68cbb5,app003,4bb702,2025-07-02 00:37:00+00:00,df_15eb7c8c,2025-07-02 10:37:00+00:00,10,7,7,NaN,20.633333,21.000000
8,chg_8f6ee15b,app001,ca3e35,2025-07-07 13:35:00+00:00,df_295ed3f1,2025-07-07 16:35:00+00:00,3,7,7,NaN,14.359496,15.000000
11,chg_90d9652e,app001,cd7d15,2025-07-10 17:58:00+00:00,df_83590aea,2025-07-13 00:58:00+00:00,55,7,7,NaN,25.557436,25.000000
40,chg_22754b79,app003,00540b,2025-07-13 02:40:00+00:00,df_d6b6cdf5,2025-07-16 01:40:00+00:00,71,7,7,NaN,35.896720,34.200000
12,chg_cfd655f7,app001,e65c95,2025-07-13 05:43:00+00:00,df_f8b93c0a,2025-07-15 07:43:00+00:00,50,7,7,NaN,38.655804,36.833333
16,chg_1918cc5e,app002,2e6951,2025-07-18 22:43:00+00:00,df_a29aae73,2025-07-19 08:43:00+00:00,10,7,7,NaN,33.699596,33.000000
14,chg_db7dbdff,app002,3d91ec,2025-07-20 13:15:00+00:00,df_8966166b,2025-07-22 04:15:00+00:00,39,7,7,NaN,34.526650,33.750000
2,chg_5e11ee73,app001,966c88,2025-07-25 05:08:00+00:00,df_45056a55,2025-07-27 12:08:00+00:00,55,7,7,NaN,37.453374,36.111111
10,chg_2c869d04,app001,4c513e,2025-07-27 19:14:00+00:00,df_325b9203,2025-07-30 06:14:00+00:00,59,7,7,NaN,40.309502,38.400000


## Figures

### ChatGPT - Lead Time for Changes (LTFC)

**Best Graph Type:** Line Chart with moving average.

**Why:** Lead time is duration measured from commit → deploy.

**It is inherently a time-series, so a line graph best shows:**

- Whether lead times are improving
- How stable or volatile the engineering process is
- The impact of process changes (e.g., new CI/CD pipeline)

**Optional secondary views:**

- Box and Whisker Plot for showing distribution of lead times (median, IQR, outliers)
- Scatter Plot per change, with trend line
- Histogram to show distribution of durations

These alternatives help illustrate variability, which is important in discussing reliability.

### Styling

['#636EFA', # the plotly blue you can see above
 '#EF553B',
 '#00CC96',
 '#AB63FA',
 '#FFA15A',
 '#19D3F3',
 '#FF6692',
 '#B6E880',
 '#FF97FF',
 '#FECB52']

### Scatter Plot

In [6]:
# https://plotly.com/python/line-and-scatter/
fig_lead_scatter = px.scatter(
	data_frame=df_lead,
	title="Lead Time for Changes Scatter Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="lead_time_hours",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_scatter.show()

### Simple Moving Average Plot

In [7]:
fig_lead_line_sma = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="SMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_line_sma.show()

### Exponential Moving Average Plot

In [20]:
fig_lead_line_ema = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="EMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_line_ema.show()

### Cumulative Moving Average Plot

In [9]:
fig_lead_line_cma = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="CMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_line_cma.show()

### Dual Scatter & MA with "traces"

In [21]:
# https://stackoverflow.com/questions/74520782/plotly-express-overlay-two-line-graphs
fig_lead_scat_trace = px.scatter(
	data_frame=df_lead,
	title="Lead Time for Changes Scatter Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="lead_time_hours",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)


fig_lead_ema_trace = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="EMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_scat_trace.add_traces(
	list(fig_lead_ema_trace.select_traces())
)


fig_lead_scat_trace.show()